# Import packages and load data

In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import ParameterGrid, cross_val_score
from sklearn.svm import SVC
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from matplotlib import rc
xscaler = MinMaxScaler()

In [2]:
file_path = '../../input_data/pspeo_master.xlsx'
model_output = 'tri'
# import data frame for pre-processing
df = pd.read_excel(file_path)
df

,sample,solv_ratio,add_type,swell_ratio,gisaxs_domain,gisaxs_fwhm,afm_domain,afm_grain,tri
0,I_A_AC,80/20,Chloronaphthalene,1.00,31.302630,0.003697,33.737303,0.0048,0
1,I_A025,80/20,Chloronaphthalene,1.25,33.805848,0.003136,35.151690,0.0047,1
2,I_A050,80/20,Chloronaphthalene,1.50,34.176062,0.002970,35.397797,0.0534,1
3,I_A075,80/20,Chloronaphthalene,1.75,34.383751,0.003451,34.603726,0.0425,1
4,I_A_100,80/20,Chloronaphthalene,2.00,31.674148,0.005122,33.786210,0.0244,1
...,...,...,...,...,...,...,...,...,...
197,Z7,80/20,Methylnaphthalene,2.50,35.569033,0.003222,37.120415,0.1093,2
198,Z8,80/20,Methylnaphthalene,2.75,35.133195,0.002910,36.469102,0.0907,2
199,Z9,80/20,Methylnaphthalene,3.00,34.889468,0.002755,33.289887,0.1990,2
200,Z10,80/20,Methylnaphthalene,3.25,36.194647,0.002669,37.443701,0.0807,2


# Data Pre-processing

In [3]:
# Assign input variables and target variable, eliminating static valuesp
inputs = df.loc[:, ['solv_ratio', 'swell_ratio', 'add_type']]
target = df['tri']

# Define mapping for solv_ratio
type_mapping = {100: 1.0, '90/10': 0.9, '80/20': 0.8, '70/30': 0.7, '60/40': 0.6, '50/50': 0.5}
inputs['solv_ratio'] = inputs['solv_ratio'].replace(type_mapping)

# Encode categorical input 'add_type'
le_addType = LabelEncoder()
inputs['add_type'] = le_addType.fit_transform(inputs['add_type'])

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(inputs, target, test_size=0.2, random_state=10)

# Scale the x data for better fitting
xscaler.fit(X_train)
X_train = xscaler.transform(X_train)
X_test = xscaler.transform(X_test)

# Grid-Search

In [4]:
%%time

# Define parameter grid for SVM
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'degree': [2, 3, 4],  # only used for 'poly' kernel
    'gamma': ['scale', 'auto'],  # or try float values
    'shrinking': [True, False]
}

# Set seed
RANDOM_SEED = 42
CV_FOLDS = 5

# Variables to store the best results
best_model = None
best_params = None
best_avg_score = float('-inf')

# Loop over all combinations
for params in ParameterGrid(param_grid):
    # Filter params if kernel is not 'poly' (avoid using 'degree')
    filtered_params = dict(params)
    if filtered_params['kernel'] != 'poly':
        filtered_params.pop('degree', None)

    model = SVC(**filtered_params, random_state=RANDOM_SEED)
    
    # Cross-validated accuracy on training set
    cv_scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring='accuracy', n_jobs=-1)
    cv_accuracy = np.mean(cv_scores)
    
    # Fit model to full training set
    model.fit(X_train, y_train)
    
    # Accuracy on test set
    test_accuracy = model.score(X_test, y_test)
    
    # Average accuracy: cross-val on train + test
    # avg_score = (cv_accuracy + test_accuracy) / 2

    if cv_accuracy > best_avg_score:
        best_avg_score = cv_accuracy
        best_model = model
        best_params = params
        best_cv_accuracy = cv_accuracy
        best_test_accuracy = test_accuracy

# Output the results
print("Best Average Accuracy (CV Train + Test):", best_avg_score)
print("CV Train Accuracy:", best_cv_accuracy)
print("Test Accuracy:", best_test_accuracy)
print("Best Parameters:", best_params)

Best Average Accuracy (CV Train + Test): 0.9316287878787879
CV Train Accuracy: 0.9316287878787879
Test Accuracy: 0.8780487804878049
Best Parameters: {'C': 10, 'degree': 2, 'gamma': 'auto', 'kernel': 'rbf', 'shrinking': True}
CPU times: user 807 ms, sys: 254 ms, total: 1.06 s
Wall time: 3.14 s


In [11]:
best = SVC(**best_params, random_state=42)
best.fit(X_train, y_train)
# test_r2 = model.score(X_test, y_test)  # or use r2_score(y_test, model.predict(X_test))

SVC(C=10, degree=2, gamma='auto', random_state=42)

In [12]:
train_r2 = best.score(X_train, y_train)
test_r2 = best.score(X_test, y_test)

print(f'Training r2: {train_r2}')
print(f'Testing r2: {test_r2}')

# Create an array for x values
x_len = len(y_test)
actual_x = np.arange(1, x_len+1)

# Define the output path for figures
output_path = f'../final_figs/{model_output}/'

Training r2: 0.9316770186335404
Testing r2: 0.8780487804878049


In [26]:
from sklearn.metrics import confusion_matrix

# Predict labels for the test set
y_pred = best.predict(X_test)

# Generate confusion matrix
cm = confusion_matrix(target, svc_fullset)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 23   3]
 [  0 124]]


In [18]:
from joblib import dump, load

dump(best, f'svc_{model_output}.joblib')

['svc_order_disorder_cv.joblib']